# NB_lab_test_trends

Builds `adb_caresync.gold.lab_test_trends` — one row per test type per calendar month with abnormal/critical rates and month-over-month trend. Full rebuild via CREATE OR REPLACE TABLE.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS

WITH monthly_agg AS (
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                                  AS year,
        MONTH(lr.test_date)                                                 AS month,
        COUNT(*)                                                            AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
    FROM adb_caresync.silver.lab_results lr
    GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),

with_rates AS (
    SELECT
        ma.*,
        CAST(ma.abnormal_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))   AS abnormal_rate_pct,
        CAST(ma.critical_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))   AS critical_rate_pct
    FROM monthly_agg ma
)

SELECT
    wr.test_name,
    wr.year,
    wr.month,
    c.month_name,
    c.quarter_name,
    wr.total_tests,
    wr.abnormal_test_count,
    wr.critical_test_count,
    wr.abnormal_rate_pct,
    wr.critical_rate_pct,
    CAST(
        LAG(wr.abnormal_rate_pct) OVER (PARTITION BY wr.test_name ORDER BY wr.year, wr.month)
    AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
    CAST(
        wr.abnormal_rate_pct
        - LAG(wr.abnormal_rate_pct) OVER (PARTITION BY wr.test_name ORDER BY wr.year, wr.month)
    AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_rates wr
LEFT JOIN adb_caresync.silver.calendar c
    ON c.date_key = CAST(DATE_FORMAT(MAKE_DATE(wr.year, wr.month, 1), 'yyyyMMdd') AS INT)

In [0]:
record_count = spark.table("adb_caresync.gold.lab_test_trends").count()
dbutils.notebook.exit(str(record_count))